LOW COMPLEXITY QPCA CODE 
SOME STUFF FROM THE GITHUB REPOSITORY THEY USED AND USING 2 X 2 MATRIX 


In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.circuit.library import phase_estimation

from typing import Tuple
import numpy as np
import scipy as sp

from qiskit import QuantumCircuit, QuantumRegister


In [2]:
from abc import ABC, abstractmethod
from typing import Tuple

from qiskit import QuantumCircuit
from qiskit.circuit.library import BlueprintCircuit


class LinearSystemMatrix(BlueprintCircuit, ABC):
    """Base class for linear system matrices."""

    def __init__(
        self,
        num_state_qubits: int,
        tolerance: float,
        evolution_time: float,
        name: str = "ls_matrix",
    ) -> None:
        """
        Args:
            num_state_qubits: the number of qubits where the unitary acts.
            tolerance: the accuracy desired for the approximation
            evolution_time: the time of the Hamiltonian simulation
            name: The name of the object.
        """
        super().__init__(name=name)

        # store parameters
        self._num_state_qubits = num_state_qubits
        self._reset_registers(num_state_qubits)
        self.tolerance = tolerance
        self.evolution_time = evolution_time

    @property
    def num_state_qubits(self) -> int:
        r"""The number of state qubits representing the state :math:`|x\rangle`.

        Returns:
            The number of state qubits.
        """
        return self._num_state_qubits

    @num_state_qubits.setter
    def num_state_qubits(self, num_state_qubits: int) -> None:
        """Set the number of state qubits.

        Note that this may change the underlying quantum register, if the number of state qubits
        changes.

        Args:
            num_state_qubits: The new number of qubits.
        """
        if num_state_qubits != self._num_state_qubits:
            self._invalidate()
            self._num_state_qubits = num_state_qubits
            self._reset_registers(num_state_qubits)

    @property
    def tolerance(self) -> float:
        """Return the error tolerance"""
        return self._tolerance

    @tolerance.setter
    def tolerance(self, tolerance: float) -> None:
        """Set the error tolerance
        Args:
            tolerance: The new error tolerance.
        """
        self._tolerance = tolerance

    @property
    def evolution_time(self) -> float:
        """Return the time of the evolution."""
        return self._evolution_time

    @evolution_time.setter
    def evolution_time(self, evolution_time: float) -> None:
        """Set the time of the evolution.

        Args:
            evolution_time: The new time of the evolution.
        """
        self._evolution_time = evolution_time

    @abstractmethod
    def eigs_bounds(self) -> Tuple[float, float]:
        """Return lower and upper bounds on the eigenvalues of the matrix."""
        raise NotImplementedError

    @abstractmethod
    def condition_bounds(self) -> Tuple[float, float]:
        """Return lower and upper bounds on the condition number of the matrix."""
        raise NotImplementedError

    @abstractmethod
    def _reset_registers(self, num_state_qubits: int) -> None:
        """Reset the registers according to the new number of state qubits.

        Args:
            num_state_qubits: The new number of qubits.
        """
        raise NotImplementedError

    @abstractmethod
    def power(self, power: int, matrix_power: bool = False) -> QuantumCircuit:
        """Build powers of the circuit.

        Args:
            power: The power to raise this circuit to.
            matrix_power: If True, the circuit is converted to a matrix and then the
                matrix power is computed. If False, and ``power`` is a positive integer,
                the implementation defaults to ``repeat``.

        Returns:
            The quantum circuit implementing powers of the unitary.
        """
        raise NotImplementedError

In [3]:
"""Hamiltonian simulation of matrices given as numpy arrays."""
#from .linear_system_matrix import LinearSystemMatrix copied the script for linear system matrix above 




class NumPyMatrix(LinearSystemMatrix):
    """Class of matrices given as a numpy array.

    Examples:

        .. jupyter-execute::

            import numpy as np
            from qiskit import QuantumCircuit
            from quantum_linear_solvers.linear_solvers.matrices.numpy_matrix import NumPyMatrix

            matrix = NumPyMatrix(np.array([[1 / 2, 1 / 6, 0, 0], [1 / 6, 1 / 2, 1 / 6, 0],
                               [0, 1 / 6, 1 / 2, 1 / 6], [0, 0, 1 / 6, 1 / 2]]))
            power = 2

            num_qubits = matrix.num_state_qubits
            # Controlled power (as used within QPE)
            pow_circ = matrix.power(power).control()
            circ_qubits = pow_circ.num_qubits
            qc = QuantumCircuit(circ_qubits)
            qc.append(matrix.power(power).control(), list(range(circ_qubits)))
    """

    def __init__(
        self,
        matrix: np.ndarray,
        tolerance: float = 1e-2,
        evolution_time: float = 1.0,
        name: str = "np_matrix",
    ) -> None:
        """
        Args:
            matrix: The matrix defining the linear system problem.
            tolerance: The accuracy desired for the approximation.
            evolution_time: The time of the Hamiltonian simulation.
            name: The name of the object.
        """

        # define internal parameters
        self._num_state_qubits = None
        self._tolerance = None
        self._evolution_time = None  # makes sure the eigenvalues are contained in [0,1)
        self._matrix = None

        super().__init__(
            num_state_qubits=int(np.log2(matrix.shape[0])),
            tolerance=tolerance,
            evolution_time=evolution_time,
            name=name,
        )

        # store parameters
        self.num_state_qubits = int(np.log2(matrix.shape[0]))
        self.tolerance = tolerance
        self.evolution_time = evolution_time
        self.matrix = matrix

    @property
    def num_state_qubits(self) -> int:
        r"""The number of state qubits representing the state :math:`|x\rangle`.

        Returns:
            The number of state qubits.
        """
        return self._num_state_qubits

    @num_state_qubits.setter
    def num_state_qubits(self, num_state_qubits: int) -> None:
        """Set the number of state qubits.

        Note that this may change the underlying quantum register, if the number of state qubits
        changes.

        Args:
            num_state_qubits: The new number of qubits.
        """
        if num_state_qubits != self._num_state_qubits:
            self._invalidate()
            self._num_state_qubits = num_state_qubits
            self._reset_registers(num_state_qubits)

    @property
    def tolerance(self) -> float:
        """Return the error tolerance"""
        return self._tolerance

    @tolerance.setter
    def tolerance(self, tolerance: float) -> None:
        """Set the error tolerance
        Args:
            tolerance: The new error tolerance.
        """
        self._tolerance = tolerance

    @property
    def evolution_time(self) -> float:
        """Return the time of the evolution."""
        return self._evolution_time

    @evolution_time.setter
    def evolution_time(self, evolution_time: float) -> None:
        """Set the time of the evolution.

        Args:
            evolution_time: The new time of the evolution.
        """
        self._evolution_time = evolution_time

    @property
    def matrix(self) -> np.ndarray:
        """Return the matrix."""
        return self._matrix

    @matrix.setter
    def matrix(self, matrix: np.ndarray) -> None:
        """Set the matrix.

        Args:
            matrix: The new matrix.
        """
        self._matrix = matrix

    def eigs_bounds(self) -> Tuple[float, float]:
        """Return lower and upper bounds on the eigenvalues of the matrix."""
        matrix_array = self.matrix
        lambda_max = max(np.abs(np.linalg.eigvals(matrix_array)))
        lambda_min = min(np.abs(np.linalg.eigvals(matrix_array)))
        return lambda_min, lambda_max

    def condition_bounds(self) -> Tuple[float, float]:
        """Return lower and upper bounds on the condition number of the matrix."""
        matrix_array = self.matrix
        kappa = np.linalg.cond(matrix_array)
        return kappa, kappa

    def _check_configuration(self, raise_on_failure: bool = True) -> bool:
        """Check if the current configuration is valid."""
        valid = True

        if self.matrix.shape[0] != self.matrix.shape[1]:
            if raise_on_failure:
                raise AttributeError("Input matrix must be square!")
            return False
        if np.log2(self.matrix.shape[0]) % 1 != 0:
            if raise_on_failure:
                raise AttributeError("Input matrix dimension must be 2^n!")
            return False
        if not np.allclose(self.matrix, self.matrix.conj().T):
            if raise_on_failure:
                raise AttributeError("Input matrix must be hermitian!")
            return False

        return valid

    def _reset_registers(self, num_state_qubits: int) -> None:
        """Reset the quantum registers.

        Args:
            num_state_qubits: The number of qubits to represent the matrix.
        """
        qr_state = QuantumRegister(num_state_qubits, "state")
        self.qregs = [qr_state]

    def _build(self) -> None:
        """If not already built, build the circuit."""
        if self._is_built:
            return

        super()._build()

        self.compose(self.power(1), inplace=True)

    def inverse(self):
        return NumPyMatrix(self.matrix, evolution_time=-1 * self.evolution_time)

    def power(self, power: int, matrix_power: bool = False) -> QuantumCircuit:
        """Build powers of the circuit.

        Args:
            power: The power to raise this circuit to.
            matrix_power: If True, the circuit is converted to a matrix and then the
                matrix power is computed. If False, and ``power`` is a positive integer,
                the implementation defaults to ``repeat``.

        Returns:
            The quantum circuit implementing powers of the unitary.
        """
        qc = QuantumCircuit(self.num_state_qubits)
        evolved = sp.linalg.expm(1j * self.matrix * self.evolution_time)
        # pylint: disable=no-member
        qc.unitary(evolved, qc.qubits)
        return qc.power(power)

In [4]:
import numpy as np

from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
#from .numpy_matrix import NumPyMatrix copied this NUMPYMATRIX INTO THE CELL ABOVE 
from qiskit.circuit.library import PhaseEstimation

class PeCircuitBuilder():
    
    @classmethod
    def generate_PE_circuit(cls,input_matrix,resolution,qram_circuit):
        """
        Generate phase estimation circuit with a number of qubits provided as resolution parameter in the constructor.

        Parameters
        ----------
        
        input_matrix: array-like of shape (n_samples, n_features)
                        Input hermitian matrix on which you want to apply QPCA, divided by its trace. Here, `n_samples` represents the number of samples,
                        and `n_features` represents the number of features. 
        
        resolution: int value
                        The number of qubits used in the phase estimation process to encode the eigenvalues.
        
        qram_circuit: QuantumCircuit 
                        The quantum circuit that encodes the input matrix.

        Returns
        ----------
        q_circuit: QuantumCircuit. 
                The quantum circuit that performs the encoding of the input matrix and Phase Estimation.
                    
        Notes
        ----------

        """
        
        u_circuit = NumPyMatrix(input_matrix, evolution_time=2*np.pi) #this evolution is what im kind of using as well but instead of NumPyMatrix i am using expm 
        pe = PhaseEstimation(resolution, u_circuit, name = "PE")
        tot_qubit = pe.qregs[0].size+qram_circuit.qregs[0].size
        qr_total = QuantumRegister(tot_qubit, 'total')
        q_circuit = QuantumCircuit(qr_total , name='matrix')
        q_circuit.append(qram_circuit.to_gate(), qr_total[pe.qregs[0].size:])
        q_circuit.append(pe.to_gate(), qr_total[0:pe.num_qubits])
        return q_circuit

In [6]:
import numpy as np

X_1 = [4,3,4,4,3,3,3,3,4,4,4,5,4,3,4]
X_2 = [3028,1365,2726,2538,1318,1693,1412,1632,2875,3564,4412,4444,4278,3064,3857]
X_1 = X_1 - np.average(X_1)
X_2 = (X_2 - np.average(X_2)) / 1000
print('The rescaled feature vectors are')
print('X_1 = ', X_1)
print('X_2 = ', X_2)

The rescaled feature vectors are
X_1 =  [ 0.33333333 -0.66666667  0.33333333  0.33333333 -0.66666667 -0.66666667
 -0.66666667 -0.66666667  0.33333333  0.33333333  0.33333333  1.33333333
  0.33333333 -0.66666667  0.33333333]
X_2 =  [ 0.21426667 -1.44873333 -0.08773333 -0.27573333 -1.49573333 -1.12073333
 -1.40173333 -1.18173333  0.06126667  0.75026667  1.59826667  1.63026667
  1.46426667  0.25026667  1.04326667]
